In [4]:
import sys, glob, torch, numpy as np
from aguirre.config import PROJECT_ROOT, TRACKED_DIR
from aguirre.Apex_net import (
    export_state_dicts, load_apex_ensemble, encode_sequences, predict_mic,
)

APEX_REPO = PROJECT_ROOT / "external" / "apex-pathogen"
BUNDLE    = TRACKED_DIR / "apex_state_dicts.pt"

export_state_dicts(APEX_REPO, BUNDLE)

# Testsequenzen aus dem Repo
from Bio import SeqIO
seqs = [str(r.seq) for r in SeqIO.parse(APEX_REPO / "test_seqs.fasta", "fasta")]

# Port
mic_port = predict_mic(load_apex_ensemble(BUNDLE, "cpu"), seqs, device="cpu")

# Original, exakt wie APEX_predict.py
sys.path.insert(0, str(APEX_REPO))
originale = [torch.load(p, map_location="cpu", weights_only=False).eval()
             for p in sorted(glob.glob(str(APEX_REPO/"APEX_pathogen_models"/"APEX_*")))]
X = torch.from_numpy(encode_sequences(seqs))
with torch.no_grad():
    mic_orig = np.mean([10.0 ** (6.0 - m(X).numpy()) for m in originale], axis=0)

print("max. relative Abweichung:",
      np.max(np.abs(mic_port - mic_orig) / np.maximum(mic_orig, 1e-12)))
assert np.allclose(mic_port, mic_orig, rtol=1e-4), "Port weicht ab!"
print("Aequivalenz bestaetigt, Shape:", mic_port.shape)

max. relative Abweichung: 0.0
Aequivalenz bestaetigt, Shape: (5, 11)


### Metal Performance Shader-Validierung

In [5]:
import numpy as np
from aguirre.apex_net import load_apex_ensemble, predict_mic

mic_cpu = predict_mic(load_apex_ensemble(BUNDLE, "cpu"), seqs, device="cpu")
mic_mps = predict_mic(load_apex_ensemble(BUNDLE, "mps"), seqs, device="mps")

abw = np.max(np.abs(mic_mps - mic_cpu) / np.maximum(mic_cpu, 1e-12))
print("max. relative Abweichung mps vs cpu:", abw)

max. relative Abweichung mps vs cpu: 3.5816106e-06
